# 04 · The transformer and the mystery of "object queries"

> **Paper:** §3.2 "Transformer encoder/decoder", Fig. 2, Fig. 7 · **Code:** [`models/transformer.py`](../models/transformer.py)

Notebook `03` left us with a sequence of `(850, 1, 256)` image tokens. Now:

1. The **encoder** lets image tokens talk to each other (global context).
2. The **decoder** takes 100 learned **object queries** and turns them into 100 detections.
3. Two small heads map each query's 256-d output to a class and a box.

The concept everyone stumbles on is the object query. We'll take it apart carefully.

## 0. Setup — everything this notebook needs

This notebook is **self-contained**: only third-party packages are imported, and every DETR-specific piece is written out below. Nothing comes from this repo, so you can read straight through without chasing a helper into another file.

Run this section once, then forget about it.

In [ ]:
# Standard third-party imports. Nothing from this repo -- every helper this
# notebook uses is defined below, in this file.
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
np.set_printoptions(precision=3, suppress=True)

ASSETS = "_assets"                      # downloaded images are cached here
os.makedirs(ASSETS, exist_ok=True)
print("torch", torch.__version__)

### COCO class names and plot colors

In [ ]:
# DETR predicts 91 "classes" + 1 no-object slot = 92 logits per query.
# COCO's category ids are not contiguous (they run 1..90 with gaps), so the gaps
# are filled with 'N/A' placeholders and index 0 is unused. The list MUST be
# exactly 91 long -- one short and every label after the gap is silently wrong.
COCO_CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush',
]
assert len(COCO_CLASSES) == 91, f"expected 91 classes, got {len(COCO_CLASSES)}"

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

### Box geometry

DETR predicts boxes as **`cxcywh`** — centre + size, normalized to `[0, 1]`. IoU and plotting want **`xyxy`** corners. Mixing the two up is the single most common bug in detection code, so both conversions live here.

In [ ]:
def box_cxcywh_to_xyxy(b):
    """(cx, cy, w, h) -> (x0, y0, x1, y1), on the last dim."""
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


def box_xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b.unbind(-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def box_area(b):
    """Area of (x0, y0, x1, y1) boxes."""
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    """Pairwise IoU. a: (N, 4), b: (M, 4), both xyxy. Returns (iou, union), each (N, M)."""
    area_a, area_b = box_area(a), box_area(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2) top-left of overlap
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])          # (N, M, 2) bottom-right
    wh = (rb - lt).clamp(min=0)                             # no overlap -> 0
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union, union


def generalized_box_iou(a, b):
    """GIoU = IoU - |C \\ (A u B)| / |C|, where C is the smallest box enclosing both.

    Unlike IoU, GIoU keeps giving gradient when the boxes do not overlap at all:
    it measures how far apart they are, in units of the enclosing box.
    Range is [-1, 1] (1 = identical, -1 = infinitely far apart).
    """
    assert (a[:, 2:] >= a[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    assert (b[:, 2:] >= b[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    iou, union = box_iou(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])          # enclosing box
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = wh[:, :, 0] * wh[:, :, 1]
    return iou - (enclosing - union) / enclosing


# --- quick self-check -------------------------------------------------------
_a = torch.tensor([[0.0, 0.0, 2.0, 2.0]])
_b = torch.tensor([[1.0, 1.0, 3.0, 3.0]])
assert torch.allclose(box_iou(_a, _b)[0], torch.tensor([[1 / 7]]))        # 1 / (4+4-1)
assert torch.allclose(generalized_box_iou(_a, _a), torch.tensor([[1.0]])) # identical -> 1
_far = torch.tensor([[10.0, 10.0, 11.0, 11.0]])
assert box_iou(_a, _far)[0].item() == 0.0                                 # IoU dies...
assert generalized_box_iou(_a, _far).item() < 0                           # ...GIoU still ranks
print("box helpers ok")

In [ ]:
class NestedTensor:
    """A padded batch of images plus the mask saying which pixels are real.

    Images in a batch have different sizes, so they are padded to a common
    (maxH, maxW). `mask[b, y, x]` is True where pixel (y, x) of image b is padding.
    """
    def __init__(self, tensors, mask):
        self.tensors, self.mask = tensors, mask

    def decompose(self):
        return self.tensors, self.mask

    def to(self, device):
        return NestedTensor(self.tensors.to(device), self.mask.to(device))


def nested_tensor_from_tensor_list(tensor_list):
    """Pad a list of (3, H_i, W_i) tensors into one batch + its padding mask."""
    max_h = max(t.shape[1] for t in tensor_list)
    max_w = max(t.shape[2] for t in tensor_list)
    b = len(tensor_list)
    dtype, device = tensor_list[0].dtype, tensor_list[0].device

    batch = torch.zeros((b, 3, max_h, max_w), dtype=dtype, device=device)
    mask = torch.ones((b, max_h, max_w), dtype=torch.bool, device=device)  # True = padding
    for img, pad_img, m in zip(tensor_list, batch, mask):
        pad_img[:, :img.shape[1], :img.shape[2]].copy_(img)
        m[:img.shape[1], :img.shape[2]] = False                            # real pixels
    return NestedTensor(batch, mask)

### Images in, tensors out

DETR's eval transform resizes the shortest side to 800px and ImageNet-normalizes. There is no fixed crop — the model accepts any input size.

In [ ]:
SAMPLE_IMAGES = {
    "cats":    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street":  "http://images.cocodataset.org/val2017/000000000139.jpg",
    "horses":  "http://images.cocodataset.org/val2017/000000006471.jpg",
    "kitchen": "http://images.cocodataset.org/val2017/000000002153.jpg",
}


def load_image(name_or_url):
    """Load a sample image by nickname, URL, or local path. Cached under _assets/."""
    url = SAMPLE_IMAGES.get(name_or_url, name_or_url)
    if os.path.exists(url):
        return Image.open(url).convert("RGB")
    path = os.path.join(ASSETS, os.path.basename(url))
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


# DETR's eval transform: resize the shortest side to 800px, to tensor, ImageNet
# normalize. There is NO fixed crop -- DETR accepts variable input sizes.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
default_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(800),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_device():
    """CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

### Drawing detections

In [ ]:
def rescale_bboxes(boxes, size):
    """Normalized cxcywh in [0,1] -> absolute xyxy pixels. `size` is PIL's (W, H)."""
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(boxes)
    return b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)


def plot_results(pil_img, prob, boxes, ax=None, title=None, linewidth=2.5):
    """prob: (n, 91) softmax WITHOUT the no-object column. boxes: (n, 4) xyxy pixels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(pil_img)
    for i, (p, (xmin, ymin, xmax, ymax)) in enumerate(zip(prob, boxes.tolist())):
        c = COLORS[i % len(COLORS)]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=linewidth))
        cl = p.argmax()
        ax.text(xmin, ymin, f'{COCO_CLASSES[cl]}: {p[cl]:0.2f}', fontsize=11,
                bbox=dict(facecolor=c, alpha=0.6, edgecolor='none'), color='white')
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax

### DETR itself

Backbone, positional encoding, transformer and prediction heads, written out. This is the same architecture as [`models/`](../models) with the inference path kept.

The proof that it is faithful is `load_state_dict(...)` below: it is **strict**, so every parameter name here has to match Facebook's released checkpoint exactly or it raises.

In [ ]:
class FrozenBatchNorm2d(nn.Module):
    """BatchNorm with statistics and affine parameters frozen as plain buffers."""
    def __init__(self, n):
        super().__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, *a, **kw):
        state_dict.pop(prefix + "num_batches_tracked", None)
        super()._load_from_state_dict(state_dict, prefix, *a, **kw)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + 1e-5).rsqrt()
        return x * scale + (b - rm * scale)


class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=128, temperature=10000, scale=2 * math.pi):
        super().__init__()
        self.num_pos_feats, self.temperature, self.scale = num_pos_feats, temperature, scale

    def forward(self, x, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class Backbone(nn.Module):
    """ResNet-50 trunk with frozen BN, returning only the last (stride-32) stage."""
    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
        self.body = torchvision.models._utils.IntermediateLayerGetter(net, {"layer4": "0"})
        self.num_channels = 2048

    def forward(self, x, mask):
        feat = self.body(x)["0"]
        feat_mask = F.interpolate(mask[None].float(), size=feat.shape[-2:]).to(torch.bool)[0]
        return feat, feat_mask


class TransformerEncoderLayer(nn.Module):
    """One encoder block: self-attention over image tokens, then a feed-forward net."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # pos is added to the QUERY and the KEY but never to the VALUE:
        # position decides where to look, not what gets carried back.
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        return self.norm2(src + self.dropout2(src2))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])

    def forward(self, src, src_key_padding_mask=None, pos=None):
        out = src
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return out


class TransformerDecoderLayer(nn.Module):
    """One decoder block: queries talk to each other, then to the image, then FFN."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]              # queries deduplicate here
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2 = self.multihead_attn(                            # queries read the image here
            query=tgt if query_pos is None else tgt + query_pos,
            key=memory if pos is None else memory + pos,
            value=memory, key_padding_mask=memory_key_padding_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        return self.norm3(tgt + self.dropout3(tgt2))


class TransformerDecoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        """Returns EVERY layer's output, stacked: (num_layers, num_queries, B, d_model).

        DETR keeps them all because the loss is applied after each decoder layer
        ("auxiliary decoding losses", paper section 3.2).
        """
        out = tgt
        intermediate = []
        for layer in self.layers:
            out = layer(out, memory, memory_key_padding_mask=memory_key_padding_mask,
                        pos=pos, query_pos=query_pos)
            intermediate.append(self.norm(out))
        return torch.stack(intermediate)


class Transformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.0):
        super().__init__()
        self.encoder = TransformerEncoder(d_model, nhead, dim_feedforward,
                                          num_encoder_layers, dropout)
        self.decoder = TransformerDecoder(d_model, nhead, dim_feedforward,
                                          num_decoder_layers, dropout)
        self.d_model, self.nhead = d_model, nhead

    def forward(self, src, mask, query_embed, pos_embed):
        """src/pos_embed: (B, C, H, W). mask: (B, H, W), True = padding."""
        bs, c, h, w = src.shape
        # (B, C, H, W) -> (H*W, B, C): this implementation puts the sequence axis first.
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
        mask = mask.flatten(1)

        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        tgt = torch.zeros_like(query_embed)                    # queries start at zero
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                          pos=pos_embed, query_pos=query_embed)
        return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k)
                                    for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < len(self.layers) - 1 else layer(x)
        return x


class DETR(nn.Module):
    def __init__(self, num_classes=91, num_queries=100, hidden_dim=256, nheads=8,
                 enc_layers=6, dec_layers=6, dim_feedforward=2048):
        super().__init__()
        self.backbone = nn.ModuleList([Backbone(),
                                       PositionEmbeddingSine(hidden_dim // 2)])
        self.transformer = Transformer(hidden_dim, nheads, enc_layers, dec_layers,
                                       dim_feedforward)
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.num_queries = num_queries

    def forward(self, images, mask=None):
        if mask is None:
            mask = torch.zeros(images.shape[0], *images.shape[-2:],
                               dtype=torch.bool, device=images.device)
        feat, feat_mask = self.backbone[0](images, mask)
        pos = self.backbone[1](feat, feat_mask)
        hs, memory = self.transformer(self.input_proj(feat), feat_mask,
                                      self.query_embed.weight, pos)
        return {"pred_logits": self.class_embed(hs)[-1],
                "pred_boxes": self.bbox_embed(hs).sigmoid()[-1]}


DETR_R50_URL = "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth"


def load_pretrained_detr(device=None):
    """Build the model above and load Facebook's released COCO weights into it.

    `load_state_dict` is strict by default, which is the real test: every parameter
    name defined above has to match the official checkpoint exactly, or this raises.
    """
    model = DETR()
    ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()
    return model.to(device) if device is not None else model


@torch.no_grad()
def detect(model, pil_img, threshold=0.9, device=None):
    """Returns (probs_kept (n, 91), boxes_kept xyxy pixels, raw outputs, keep mask)."""
    device = device or next(model.parameters()).device
    x = default_transform(pil_img).unsqueeze(0).to(device)
    outputs = model(x)
    probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # drop no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), pil_img.size)
    return probs[keep], boxes, outputs, keep

In [ ]:
device = get_device()
model = load_pretrained_detr(device=device)
im = load_image("cats")
single = nested_tensor_from_tensor_list([default_transform(im)]).to(device)
print("device:", device, "| tokens in:", tuple(single.tensors.shape))

## 1. The encoder: 850 tokens, all looking at each other

The encoder is 6 standard transformer layers. Each does **self-attention over the image tokens**, so after the encoder every position "knows" about the whole image.

Shapes are unchanged throughout — `(850, 1, 256)` in, `(850, 1, 256)` out. An encoder *refines* tokens; it never changes how many there are.

In [ ]:
print(model.transformer.encoder.layers[0])

In [ ]:
with torch.no_grad():
    s, m = model.backbone[0](single.tensors, single.mask)       # ResNet trunk
    pos2d = model.backbone[1](s, m)                              # sine positional encoding
    src = model.input_proj(s).flatten(2).permute(2, 0, 1)        # (HW, B, C) = (850, 1, 256)
    pos = pos2d.flatten(2).permute(2, 0, 1)                      # (HW, B, C)
    mask = m.flatten(1)                                          # (B, HW)    = (1, 850)
    memory = model.transformer.encoder(src, src_key_padding_mask=mask, pos=pos)

print("encoder input  :", tuple(src.shape))
print("encoder output :", tuple(memory.shape), "  <- 'memory', same shape")
print()
print("attention matrix per layer per head:",
      f"({src.shape[0]} x {src.shape[0]}) = {src.shape[0]**2:,} weights")
print("layers:", len(model.transformer.encoder.layers), " heads:", model.transformer.nhead)

Why bother? The paper's ablation (**Table 2**) removes the encoder entirely: AP falls from 40.6 to 36.7, and **−6.0 AP on large objects**. Their explanation (§4.2):

> *"The encoder seems to separate instances already, which likely simplifies object extraction and localization for the decoder."*

A large object spans many grid cells; only global attention can bind them into a single thing. We'll *see* this happening in notebook `06`.

## 2. Object queries: 100 learned vectors, and what they are not

Here is the entire definition, from [`models/detr.py:39`](../models/detr.py#L39):

```python
self.query_embed = nn.Embedding(num_queries, hidden_dim)   # (100, 256)
```

That's it. **100 vectors of 256 numbers, learned by gradient descent, then frozen at inference.** They do not depend on the image in any way.

In [ ]:
q = model.query_embed.weight            # (num_queries, d_model) = (100, 256)
print("query_embed.weight:", tuple(q.shape))
print("depends on the input image?  No -- it is an nn.Parameter, identical for every photo.")
print()
print("first query, first 8 dims:", [round(v, 3) for v in q[0, :8].tolist()])
print("norms: min %.2f  max %.2f  mean %.2f" % (q.norm(dim=1).min(), q.norm(dim=1).max(), q.norm(dim=1).mean()))

### The clearest way to think about it

A query is a **standing question** the model asks about every image. Loosely: *"is there a smallish object in the lower-left?"* Query 0 asks its question, query 1 asks a different one, and so on 100 times — **in parallel**, all in one forward pass.

Common misconceptions, cleared up:

| Myth | Reality |
|---|---|
| "A query is a region proposal" | No — it carries no coordinates, and is the same for every image |
| "Query *k* always detects class *k*" | No — Fig. 5 shows one model finding 24 giraffes; slots aren't class-bound |
| "Queries are generated from the image" | No — they're `nn.Parameter`s, fixed after training |
| "They're like NLP decoder tokens" | Closer, but NLP decodes **autoregressively**, one token at a time. DETR decodes all 100 **at once** |

### The odd bit: the decoder's input is all zeros

Look at [`models/transformer.py:55`](../models/transformer.py#L55):

In [ ]:
# Transformer.forward is defined in section 0. Rather than print it again, watch the
# shapes move through it -- that is the part that is hard to hold in your head.
stages = [
    ("feature map from the backbone", tuple(s.shape)),
    ("after input_proj (1x1 conv)",   tuple(model.input_proj(s).shape)),
    ("flattened to a sequence",       tuple(src.shape)),
    ("  positional encoding",         tuple(pos.shape)),
    ("  padding mask",                tuple(mask.shape)),
    ("encoder output (memory)",       tuple(memory.shape)),
    ("query embeddings",              tuple(model.query_embed.weight.shape)),
]
print(f"{'stage':<32} shape")
print("-" * 58)
for k, v in stages:
    print(f"{k:<32} {v}")
print()
print("Note the sequence axis comes FIRST: (S, B, C), not (B, S, C).")
print("This implementation predates batch_first=True being common.")

`tgt = torch.zeros_like(query_embed)` — the decoder's *content* stream starts at **zero**, and `query_embed` is passed separately as `query_pos`, added to the query and key at every attention layer (exactly like the spatial positional encoding in notebook `03`).

So an object query is a **positional encoding for output slots**. The paper calls them *"learnt positional encodings that we refer to as object queries"* (§3.2). All content comes from attending to the image; the query only decides *what to go looking for*.

Why must the 100 be different from each other? Because self-attention is permutation-invariant: *"since the decoder is also permutation-invariant, the N input embeddings must be different to produce different results."* Identical queries would produce 100 identical boxes.

## 3. Running the decoder

Each decoder layer does three things, in order:

1. **Self-attention among the 100 queries** — how they avoid duplicating each other
2. **Cross-attention into encoder memory** — how they actually look at the image
3. **FFN** — per-query processing

In [ ]:
print(model.transformer.decoder.layers[0])

In [ ]:
with torch.no_grad():
    query_embed = model.query_embed.weight.unsqueeze(1)   # (num_queries, B=1, d_model)
    tgt = torch.zeros_like(query_embed)                   # queries start at ZERO
    hs = model.transformer.decoder(tgt, memory, memory_key_padding_mask=mask,
                                   pos=pos, query_pos=query_embed)

print("queries in :", tuple(tgt.shape),    " (all zeros)")
print("memory     :", tuple(memory.shape), " (850 image tokens)")
print("decoder out:", tuple(hs.shape),     " <- (n_layers, 100, B, 256)")
print()
print("All 6 layers come back because TransformerDecoder.forward stacks them:")
print("  the training loss is applied after EVERY layer ('auxiliary decoding losses')")
print()
print("The queries carry no information at all at the start -- everything they")
print("become comes from query_pos (the learned slots) and the image.")

Note the shape `(6, 100, 1, 256)`: the decoder hands back its output after **each** of the 6 layers, not just the last. That's for the auxiliary losses. At inference only `hs[-1]` is used — see [`models/detr.py:69`](../models/detr.py#L69), `outputs_class[-1]`.

## 4. Two heads turn 256-d vectors into detections

In [ ]:
hs_ = hs.transpose(1, 2)               # (6, B, 100, 256) -- how detr.py sees it
print("hs:", tuple(hs_.shape))

with torch.no_grad():
    outputs_class = model.class_embed(hs_)            # (layers, B, queries, classes+1)
    outputs_coord = model.bbox_embed(hs_).sigmoid()   # (layers, B, queries, 4)

print("class_embed -> :", tuple(outputs_class.shape), "  Linear(256 -> 91+1)")
print("bbox_embed  -> :", tuple(outputs_coord.shape), "  MLP(256->256->256->4) then sigmoid")
print()
print("bbox_embed is a 3-layer MLP:")
for i, l in enumerate(model.bbox_embed.layers):
    print(f"   layer {i}: {l}")
print()
print("sigmoid guarantees cx,cy,w,h land in [0,1]:")
print("   min %.4f  max %.4f" % (outputs_coord.min(), outputs_coord.max()))

Why an MLP for boxes but a plain `Linear` for classes? Class is a linear readout of a well-separated 256-d space. Box regression needs to *compute* geometry from the embedding — non-linearity helps. Paper: *"The final prediction is computed by a 3-layer perceptron with ReLU... and a linear projection layer"* (§3.2).

### Verify against the real forward pass

In [ ]:
with torch.no_grad():
    ref = model(single.tensors, single.mask)
print("hand-rolled vs model():")
print("  logits match:", torch.allclose(outputs_class[-1], ref["pred_logits"], atol=1e-4))
print("  boxes  match:", torch.allclose(outputs_coord[-1], ref["pred_boxes"],  atol=1e-4))

## 5. Predictions sharpen with every decoder layer

Because all 6 layers are supervised, we can decode *each* and watch the model refine its answer. This reproduces the trend in the paper's **Fig. 4** (AP climbs +8.2 from layer 1 to layer 6).

In [ ]:
for layer in range(6):
    probs_l = outputs_class[layer, 0].softmax(-1)[:, :-1].cpu()   # (queries, classes) = (100, 91)
    conf = probs_l.max(-1).values
    n = (conf > 0.9).sum().item()
    labels = sorted({COCO_CLASSES[c] for c in probs_l[conf > 0.9].argmax(-1).tolist()})
    print(f"  decoder layer {layer+1}: {n:2d} confident detections   {labels}")

Layer 1 is typically over-eager (duplicates, spurious boxes); by layer 6 the set has settled. The paper notes that NMS *helps* after layer 1 but *hurts* after layer 6 — by then query self-attention has already removed the duplicates that NMS exists to clean up.

## 6. Do queries specialize? (reproducing Fig. 7)

The paper plots, for each query slot, the centers of every box it predicts across the whole val set. Each slot turns out to own a region and a size regime. Let's do a small version over a handful of images.

In [ ]:
names = list(SAMPLE_IMAGES)
all_boxes = []
with torch.no_grad():
    for n in names:
        img = load_image(n)
        o = model(default_transform(img).unsqueeze(0).to(device))
        all_boxes.append(o["pred_boxes"][0].cpu())      # (queries, cxcywh) = (100, 4)
boxes_stack = torch.stack(all_boxes)                     # (n_images, 100, 4)
print("collected:", tuple(boxes_stack.shape), "= (images, queries, cxcywh)")

In [ ]:
slots = [0, 7, 12, 25, 40, 63, 77, 99]
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax, k in zip(axes.ravel(), slots):
    b = boxes_stack[:, k, :]                             # (n_images, 4)
    sizes = (b[:, 2] * b[:, 3]).clamp(0, 1)              # normalized area
    ax.scatter(b[:, 0], b[:, 1], c=sizes, cmap="viridis",
               vmin=0, vmax=1, s=90, edgecolors="k", linewidths=.5)
    ax.set_xlim(0, 1); ax.set_ylim(1, 0)                 # y inverted = image coords
    ax.set_title(f"query slot {k}", fontsize=10)
    ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1]); ax.tick_params(labelsize=7)
plt.suptitle("Box centers predicted by individual query slots (color = box area)")
plt.tight_layout(); plt.show()

With only 4 images this is a faint echo of Fig. 7, but the point stands: **different slots sit in different places.** Some cluster in a corner; some park at (0.5, 0.5) with a large area — the paper's observation that *"almost all slots have a mode of predicting large image-wide boxes that are common in COCO"*.

To see it properly you'd run this over COCO val (5000 images). Try it if you download the dataset.

## What's next

`05` is the heart of the paper: with 100 predictions and (say) 5 ground-truth objects, **how do you even define a loss?**

## Exercises

**Exercise 1.** At `threshold=0.5`, count predictions per decoder layer. Does layer 1 produce duplicates?

<details><summary>Solution</summary>

```python
for layer in range(6):
    pl = outputs_class[layer, 0].softmax(-1)[:, :-1].cpu()
    print(f"layer {layer+1}: {(pl.max(-1).values > 0.5).sum().item():2d} predictions")
```

Layer 1 typically fires on more slots than layer 6 — several of them near-duplicates on the same object. By layer 6 the decoder's *query self-attention* has had six chances to let queries compare notes and suppress each other.

This is the paper's Fig. 4 result: NMS **improves** layer-1 output but **hurts** layer-6 output, because by then there is nothing left to suppress and NMS only deletes true positives.
</details>

---

**Exercise 2.** Zero out the object queries and re-run detection. Explain the result using notebook `01`.

<details><summary>Solution</summary>

```python
saved = model.query_embed.weight.data.clone()
model.query_embed.weight.data.zero_()
with torch.no_grad():
    o = model(default_transform(im).unsqueeze(0).to(device))
b = o["pred_boxes"][0].cpu()
print("unique boxes among 100 queries:", len(torch.unique(b.round(decimals=4), dim=0)))
model.query_embed.weight.data = saved          # IMPORTANT: undo it
```

You get **1** unique box. All 100 queries become identical vectors, and attention is permutation-invariant (notebook `01` §5), so identical inputs must give identical outputs — 100 copies of one prediction.

This is the experimental proof of §3.2's *"the N input embeddings must be different to produce different results."*
</details>

---

**Exercise 3.** The decoder's content input is `tgt = torch.zeros_like(query_embed)`. Why zeros, and not the queries themselves?

<details><summary>Solution</summary>

Because the query is a **positional** encoding, not content. The split is deliberate:

- `tgt` (content) starts empty — the decoder has no information about this image yet.
- `query_pos` is added to Q and K at every layer, steering *where each slot looks*.

All actual content arrives via cross-attention from the encoder memory. If you seeded `tgt` with the query vectors, position would leak into the value stream — the same mistake as adding positional encodings to V (notebook `01`, Exercise 4).
</details>

---

**Exercise 4.** Plot slot specialization over more slots (`range(0, 100, 12)`). Any slot look unused?

<details><summary>Solution</summary>

```python
slots = list(range(0, 100, 12))
fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for ax, k in zip(axes.ravel(), slots):
    b = boxes_stack[:, k, :]
    ax.scatter(b[:, 0], b[:, 1], c=(b[:, 2]*b[:, 3]).clamp(0, 1),
               cmap="viridis", vmin=0, vmax=1, s=80, edgecolors="k", linewidths=.5)
    ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_title(f"slot {k}", fontsize=9)
plt.tight_layout(); plt.show()
```

With only 4 images you can't really call a slot unused — you'd need COCO val (5000 images) to see Fig. 7's structure. What you *can* see is that slots occupy different regions, and that many have a mode near the center with large area (the "image-wide box" mode the paper notes).
</details>